# Evaluator Module
The Evaluator module creates evaluation reports.

Reports contain evaluation metrics depending on models specified in the evaluation config.

In [1]:
# reloads modules automatically before entering the execution of code
%load_ext autoreload
%autoreload 2

# third parties imports
import numpy as np 
import pandas as pd
# -- add new imports here --

# local imports
from configs import EvalConfig
from constants import Constant as C
from loaders import export_evaluation_report
from loaders import load_ratings
# -- add new imports here --
from surprise import accuracy
from surprise.model_selection import cross_validate, train_test_split, LeaveOneOut
from models import *
import random as rd

# 1. Try the loader with surprise_format set to True
data = load_ratings(surprise_format=True)

# 2. Verify the results
print(f"Object Type: {type(data)}")

# 3. Build a trainset to confirm data is correctly loaded (check)
trainset = data.build_full_trainset()
print(f"Number of ratings: {trainset.n_ratings}")
print(f"Number of users: {trainset.n_users}")
print(f"Number of items: {trainset.n_items}")

print("Loader successfully tested in Surprise format!")


Object Type: <class 'surprise.dataset.DatasetAutoFolds'>
Number of ratings: 381181
Number of users: 1000
Number of items: 8737
Loader successfully tested in Surprise format!


# 1. Model validation functions
Validation functions are a way to perform crossvalidation on recommender system models. 

In [2]:
def generate_split_predictions(algo, ratings_dataset, eval_config):
    """Generate predictions on a random test set specified in eval_config"""
    trainset, testset = train_test_split(
        ratings_dataset,
        test_size=eval_config.test_size,
        random_state=42
    )
    algo.fit(trainset)
    predictions = algo.test(testset)
    return predictions


def generate_loo_top_n(algo, ratings_dataset, eval_config):
    """Generate top-n recommendations for each user on a random Leave-one-out split (LOO)"""
    loo = LeaveOneOut(n_splits=1, random_state=1)
    for trainset, testset in loo.split(ratings_dataset):
        algo.fit(trainset)
        anti_testset = trainset.build_anti_testset()
        predictions = algo.test(anti_testset)
        anti_testset_top_n = get_top_n(predictions, n=eval_config.top_n_value)
    return anti_testset_top_n, testset


# ---------------------------------------------------------------------------
# Negative-sampling LOO -- protocole litterature (He et al. 2017 NCF, WWW'17)
# Rank the hidden item among 100 candidates (1 pos + 99 random negatives).
# Note (Krichene & Rendle, NeurIPS'20): sampled metrics differ from exact
# metrics -- use for internal model comparison and relative positioning
# w.r.t. MF baselines (BPR-MF HR@10~0.66, eALS HR@10~0.68 on ML-1M).
# ---------------------------------------------------------------------------

def sample_negative_anti_testset(trainset, testset, n_negatives=99, random_state=42):
    """Build a reduced anti-testset using negative sampling (literature standard).

    For each user in the testset, sample n_negatives items the user has NOT rated,
    then add the 1 hidden positive item (LOO item). Yields 100 candidates per user.

    Protocol used in:
      - He et al. (2017) Neural Collaborative Filtering, WWW '17
      - Kang & McAuley (2018) SASRec, ICDM '18
      - He et al. (2020) LightGCN, SIGIR '20
    """
    rng = np.random.default_rng(random_state)
    fill = trainset.global_mean

    user_rated = {uid: set(iid for iid, _ in trainset.ur[uid])
                  for uid in range(trainset.n_users)}
    all_items = set(range(trainset.n_items))

    anti_testset = []
    for raw_uid, raw_iid, _ in testset:
        try:
            uid = trainset.to_inner_uid(raw_uid)
        except ValueError:
            continue

        try:
            pos_inner = trainset.to_inner_iid(raw_iid)
        except ValueError:
            pos_inner = None

        rated = user_rated[uid]
        candidates = list(all_items - rated)
        if pos_inner is not None and pos_inner in candidates:
            candidates.remove(pos_inner)

        n_sample = min(n_negatives, len(candidates))
        neg_inner = rng.choice(candidates, size=n_sample, replace=False).tolist()

        for iid in neg_inner:
            anti_testset.append((raw_uid, trainset.to_raw_iid(iid), fill))

        # Always add the positive (hidden) item
        anti_testset.append((raw_uid, raw_iid, fill))

    return anti_testset


def generate_loo_neg_sampling_top_n(algo, ratings_dataset, eval_config, n_negatives=99):
    """Generate top-n recommendations using negative sampling (literature protocol).

    Same LOO split (random_state=1) as generate_loo_top_n for consistency.
    Ranks the hidden item among 100 candidates (1 positive + 99 random negatives).
    """
    loo = LeaveOneOut(n_splits=1, random_state=1)
    for trainset, testset in loo.split(ratings_dataset):
        algo.fit(trainset)
        anti_testset = sample_negative_anti_testset(
            trainset, testset, n_negatives=n_negatives
        )
        predictions = algo.test(anti_testset)
        anti_testset_top_n = get_top_n(predictions, n=eval_config.top_n_value)
    return anti_testset_top_n, testset


def generate_full_top_n(algo, ratings_dataset, eval_config, precomputed_dict=None):
    """Generate top-n recommendations for each user with full training set"""
    full_trainset = ratings_dataset.build_full_trainset()
    algo.fit(full_trainset)
    anti_testset = full_trainset.build_anti_testset()
    predictions = algo.test(anti_testset)
    #anti_testset_top_n = get_top_n(predictions, n=eval_config.top_n_value)
    #return anti_testset_top_n
    # Reranking
    if (
    getattr(algo, "rerank_popularity", False)
    and precomputed_dict
    and "item_freq" in precomputed_dict
):
        item_freq = precomputed_dict["item_freq"]
        return get_top_n_reranked(
            predictions,
            item_freq,
            alpha=getattr(algo, "rerank_alpha", 0.1),
            n=eval_config.top_n_value
        )

    return get_top_n(predictions, n=eval_config.top_n_value)


def precompute_information(df_ratings):
    """Returns a dictionary of precomputed information used by full-mode metrics.

    Keys:
    - item_to_rank   : {movie_id: popularity rank}  -- used by novelty (rang)
    - n_users        : total number of unique users  -- used by MIUF
    - item_freq      : {movie_id: nb users who rated it} -- used by MIUF
    - genre_vectors  : {movie_id: normalized genre vector} -- used by ILD
    """
    from loaders import load_items
    precomputed_dict = {}

    # item_to_rank (novelty rang)
    movie_counts = df_ratings[C.ITEM_ID_COL].value_counts()
    precomputed_dict["item_to_rank"] = movie_counts.rank(ascending=False, method='first').to_dict()

    # n_users + item_freq (MIUF)
    precomputed_dict["n_users"]    = df_ratings[C.USER_ID_COL].nunique()
    precomputed_dict["item_freq"]  = (
        df_ratings.groupby(C.ITEM_ID_COL)[C.USER_ID_COL].nunique().to_dict()
    )

    # genre_vectors (ILD) -- build_genre_vectors is defined in the metrics cell below
    df_items = load_items()
    precomputed_dict["genre_vectors"] = build_genre_vectors(df_items)

    return precomputed_dict


def create_evaluation_report(eval_config, sp_ratings, precomputed_dict, available_metrics):
    """Create a DataFrame evaluating various models on metrics specified in an evaluation config."""
    evaluation_dict = {}

    # neg_sampling config (optional attributes with safe defaults)
    neg_sampling_metrics     = getattr(eval_config, 'neg_sampling_metrics', [])
    neg_sampling_model_names = getattr(eval_config, 'neg_sampling_model_names', None)

    for model_name, model, arguments in eval_config.models:
        print(f'Handling model {model_name}')
        args = dict(arguments)
        rerank_pop   = args.pop("rerank_popularity", False)
        rerank_alpha = args.pop("rerank_alpha", 0.1)
        algo = model(**args)
        algo.rerank_popularity = rerank_pop
        algo.rerank_alpha      = rerank_alpha
        evaluation_dict[model_name] = {}

        if len(eval_config.split_metrics) > 0:
            print('  Training split predictions')
            predictions = generate_split_predictions(algo, sp_ratings, eval_config)
            for metric in eval_config.split_metrics:
                print(f'  - computing metric {metric}')
                assert metric in available_metrics['split']
                evaluation_function, parameters = available_metrics["split"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(predictions, **parameters)

        # Reranking 
        if len(eval_config.loo_metrics) > 0:
            print('Training loo predictions')
            
            # 1.
            from surprise.model_selection import LeaveOneOut
            loo = LeaveOneOut(n_splits=1, random_state=1)
            
            for trainset, testset in loo.split(sp_ratings):
                algo.fit(trainset)
                anti_testset = trainset.build_anti_testset()
                # Génération des scores bruts sans couper au Top-N immédiatement
                raw_predictions = algo.test(anti_testset)
            
            # 2. Extracting item frequency distribution for re-ranking (if needed)
            item_freq_distribution = precomputed_dict.get("item_freq", {})
            
            # 3. Reranking applied to the raw predictions before computing metrics
            if getattr(algo, "rerank_popularity", False):
                anti_testset_top_n = get_top_n_reranked(
                    raw_predictions,
                    item_freq=item_freq_distribution,
                    alpha=getattr(algo, "rerank_alpha", 0.1),
                    n=eval_config.top_n_value
                )
            else:
                anti_testset_top_n = get_top_n(
                    raw_predictions,
                    n=eval_config.top_n_value
                )
            
            # 4. Computing the ranking metrics
            for metric in eval_config.loo_metrics:
                assert metric in available_metrics['loo']
                evaluation_function, parameters =  available_metrics["loo"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(anti_testset_top_n, testset, **parameters)

        #if len(eval_config.loo_metrics) > 0:
            #print('  Training loo predictions (full catalogue)')
            #anti_testset_top_n, testset = generate_loo_top_n(algo, sp_ratings, eval_config)
            #for metric in eval_config.loo_metrics:
                #assert metric in available_metrics['loo']
                #evaluation_function, parameters = available_metrics["loo"][metric]
                #evaluation_dict[model_name][metric] = evaluation_function(anti_testset_top_n, testset, **parameters)


        # --- Negative-sampling LOO (protocole He et al. 2017 : 1 pos + 99 neg) ---
        run_ns = (
            len(neg_sampling_metrics) > 0
            and (neg_sampling_model_names is None or model_name in neg_sampling_model_names)
        )
        if run_ns:
            print(f'  Training loo predictions (negative sampling, 100 candidates)')
            anti_testset_ns, testset_ns = generate_loo_neg_sampling_top_n(algo, sp_ratings, eval_config)
            for metric in neg_sampling_metrics:
                assert metric in available_metrics['loo']
                evaluation_function, parameters = available_metrics["loo"][metric]
                col_name = f"{metric}[ns]"   # [ns] = negative sampling
                evaluation_dict[model_name][col_name] = evaluation_function(
                    anti_testset_ns, testset_ns, **parameters
                )

        if len(eval_config.full_metrics) > 0:
            print('  Training full predictions')
            anti_testset_top_n = generate_full_top_n(algo, sp_ratings, eval_config, precomputed_dict=precomputed_dict)
            for metric in eval_config.full_metrics:
                assert metric in available_metrics['full']
                evaluation_function, parameters = available_metrics["full"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(
                    anti_testset_top_n,
                    **precomputed_dict,
                    **parameters
                )

    return pd.DataFrame.from_dict(evaluation_dict).T

# 2. Evaluation metrics
Implement evaluation metrics for either rating predictions (split metrics) or for top-n recommendations (loo metric, full metric)

In [3]:
def get_hit_rate(anti_testset_top_n, testset):
    """Compute the average hit over the users (loo metric) — uses full top-N list."""
    hits = 0
    total_users = len(testset)
    for user_id, movie_id, _ in testset:
        if user_id in anti_testset_top_n:
            recommendations = [item_id for (item_id, _) in anti_testset_top_n[user_id]]
            if movie_id in recommendations:
                hits += 1
    return hits / total_users if total_users > 0 else 0


def get_hit_rate_at_k(anti_testset_top_n, testset, k):
    """Average Hit Rate@k over users (loo metric).
    HR@k = 1 if the hidden item appears in the top-k recommendations, else 0.
    Source: He et al. (2017) Neural Collaborative Filtering, WWW '17.
            Kang & McAuley (2018) SASRec, ICDM '18.
    """
    hits = 0
    n_users = 0
    for user_id, movie_id, _ in testset:
        if user_id in anti_testset_top_n:
            recs = [item_id for item_id, _ in anti_testset_top_n[user_id]]
            if movie_id in recs[:k]:
                hits += 1
            n_users += 1
    return hits / n_users if n_users > 0 else 0.0


def get_novelty(anti_testset_top_n, item_to_rank, **kwargs):
    """Average popularity rank of recommended items (full metric). Higher = more novel."""
    total_novelty = 0
    total_users = len(anti_testset_top_n)
    for user_id, recommendations in anti_testset_top_n.items():
        user_sum = sum(item_to_rank.get(movie_id, len(item_to_rank)) for movie_id, _ in recommendations)
        total_novelty += user_sum
    return total_novelty / total_users if total_users > 0 else 0.0


def get_ndcg_at_k(anti_testset_top_n, testset, k):
    """Average NDCG@k over users (loo metric).
    With one hidden item per user (LOO): NDCG@k = 1/log2(rank+1) if rank <= k, else 0.
    Source: He et al. (2017) NCF ; He et al. (2020) LightGCN.
    """
    total = 0.0
    n_users = 0
    for user_id, movie_id, _ in testset:
        if user_id in anti_testset_top_n:
            recs = [item_id for item_id, _ in anti_testset_top_n[user_id]]
            top_k = recs[:k]
            if movie_id in top_k:
                rank = top_k.index(movie_id) + 1
                total += 1.0 / np.log2(rank + 1)
            n_users += 1
    return total / n_users if n_users > 0 else 0.0


def get_precision_at_k(anti_testset_top_n, testset, k):
    """Average Precision@k over users (loo metric).
    With one hidden item per user (LOO): Precision@k = 1/k if item in top-k, else 0.
    Note: redondant avec HR@k en LOO single-item (= HR@k / k).
    """
    total = 0.0
    n_users = 0
    for user_id, movie_id, _ in testset:
        if user_id in anti_testset_top_n:
            recs = [item_id for item_id, _ in anti_testset_top_n[user_id]]
            if movie_id in recs[:k]:
                total += 1.0 / k
            n_users += 1
    return total / n_users if n_users > 0 else 0.0


def get_recall_at_k(anti_testset_top_n, testset, k):
    """Average Recall@k over users (loo metric).
    With one hidden item per user (LOO): Recall@k = 1 if item in top-k, else 0.
    Note: redondant avec HR@k en LOO single-item (identique).
    """
    hits = 0
    n_users = 0
    for user_id, movie_id, _ in testset:
        if user_id in anti_testset_top_n:
            recs = [item_id for item_id, _ in anti_testset_top_n[user_id]]
            if movie_id in recs[:k]:
                hits += 1
            n_users += 1
    return hits / n_users if n_users > 0 else 0.0


def get_coverage(anti_testset_top_n, item_to_rank, **kwargs):
    """Catalogue coverage (full metric).
    Coverage = |unique recommended items| / |total catalogue|
    """
    recommended_items = set()
    for recommendations in anti_testset_top_n.values():
        for item_id, _ in recommendations:
            recommended_items.add(item_id)
    total_items = len(item_to_rank)
    return len(recommended_items) / total_items if total_items > 0 else 0.0


def get_novelty_miuf(anti_testset_top_n, n_users, item_freq, **kwargs):
    """Mean Inverse User Frequency novelty (full metric).
    MIUF = (1/|R|) * sum_{i in R} -log2(|U_i| / |U|)
    Higher = recommends less popular items.
    Source: Vargas & Castells (2011), RecSys '11.
    """
    user_novelties = []
    for uid, recs in anti_testset_top_n.items():
        if not recs:
            continue
        miuf_scores = [-np.log2(item_freq.get(iid, 1) / n_users) for iid, _ in recs]
        user_novelties.append(np.mean(miuf_scores))
    return float(np.mean(user_novelties)) if user_novelties else 0.0


def build_genre_vectors(df_items):
    """Build normalized binary genre vectors per item (used for ILD)."""
    all_genres = sorted(set(
        g for genres in df_items[C.GENRES_COL].fillna('').str.split('|')
        for g in genres if g and g != '(no genres listed)'
    ))
    genre_index = {g: i for i, g in enumerate(all_genres)}
    vectors = {}
    for mid, row in df_items.iterrows():
        vec = np.zeros(len(all_genres))
        for g in str(row[C.GENRES_COL]).split('|'):
            if g in genre_index:
                vec[genre_index[g]] = 1.0
        norm = np.linalg.norm(vec)
        vectors[mid] = vec / norm if norm > 0 else vec
    return vectors


def get_diversity_ild(anti_testset_top_n, genre_vectors, **kwargs):
    """Intra-List Diversity (full metric).
    ILD = (1 / |R|(|R|-1)) * sum_{i!=j} (1 - cos(g_i, g_j))
    Higher = more diverse recommendations across genres.
    Source: Ziegler et al. (2005), WWW '05 ; Kunaver & Pozrl (2017), ACM TiiS.
    """
    user_ilds = []
    for uid, recs in anti_testset_top_n.items():
        ids = [iid for iid, _ in recs if iid in genre_vectors]
        R = len(ids)
        if R < 2:
            continue
        vecs = np.array([genre_vectors[i] for i in ids])
        sim = vecs @ vecs.T
        total_dist = np.sum(1 - sim)
        user_ilds.append(total_dist / (R * (R - 1)))
    return float(np.mean(user_ilds)) if user_ilds else 0.0

## Pitfalls of Using a Sum of Ranks as a Novelty Metric

1. **Sensitivity to N (top-n size)**: If two models recommend different numbers
   of items (e.g. 10 vs 40), the sum of ranks will be mechanically higher for
   the one recommending more items, without being truly more "novel".

2. **Linearity of ranks**: The difference between rank 1 and rank 100 is
   treated the same as between rank 1000 and rank 1100. However, moving from
   the most popular movie to the 100th is a far more radical shift in popularity
   than moving from rank 1000 to 1100. A logarithmic scale would better capture
   this reality.

3. **Quality vs. Novelty trade-off**: A model could achieve a very high novelty
   score by recommending the least popular (and potentially worst) movies on the
   platform that nobody wants to watch. Novelty should always be balanced with
   precision metrics (MAE, RMSE, Hit Rate) to ensure recommendations remain
   relevant.

4. **Dependence on catalogue size**: A rank of 500 in a catalogue of 600 movies
   does not have the same meaning as a rank of 500 in a catalogue of 100,000
   movies. The metric is therefore not comparable across systems with catalogues
   of different sizes.

# 3. Evaluation workflow
Load data, evaluate models and save the experimental outcomes

In [4]:
from surprise import Dataset, Reader
np.random.seed(1)
rd.seed(1)

AVAILABLE_METRICS = {
    "split": {
        "mae":  (accuracy.mae,  {'verbose': False}),
        "rmse": (accuracy.rmse, {'verbose': False}),
    },
    "loo": {
        # Hit Rate@k — He et al. (2017) NCF ; Kang & McAuley (2018) SASRec
        "hit_rate@5":   (get_hit_rate_at_k, {"k": 5}),
        "hit_rate@10":  (get_hit_rate_at_k, {"k": 10}),
        "hit_rate@20":  (get_hit_rate_at_k, {"k": 20}),
        # NDCG@k — He et al. (2017) NCF ; He et al. (2020) LightGCN
        "ndcg@5":       (get_ndcg_at_k,     {"k": 5}),
        "ndcg@10":      (get_ndcg_at_k,     {"k": 10}),
        "ndcg@20":      (get_ndcg_at_k,     {"k": 20}),
        # Kept for compatibility (redundant in single-item LOO)
        "hit_rate":     (get_hit_rate,       {}),
        "precision@5":  (get_precision_at_k, {"k": 5}),
        "precision@10": (get_precision_at_k, {"k": 10}),
        "precision@20": (get_precision_at_k, {"k": 20}),
        "recall@5":     (get_recall_at_k,    {"k": 5}),
        "recall@10":    (get_recall_at_k,    {"k": 10}),
        "recall@20":    (get_recall_at_k,    {"k": 20}),
    },
    "full": {
        # Coverage — standard catalogue coverage
        "coverage": (get_coverage,      {}),
        # MIUF — Vargas & Castells (2011), formule -log2(|U_i|/|U|)
        "miuf":     (get_novelty_miuf,  {}),
        # ILD — Ziegler et al. (2005) ; Kunaver & Pozrl (2017)
        "ild":      (get_diversity_ild, {}),
        # Novelty rank (kept for compatibility)
        "novelty":  (get_novelty,       {}),
    },
}

#df_ratings_pd = load_ratings(surprise_format=False)
#sp_ratings = load_ratings(surprise_format=True)
#precomputed_dict = precompute_information(df_ratings_pd)
#evaluation_report = create_evaluation_report(EvalConfig(), sp_ratings, precomputed_dict, AVAILABLE_METRICS)
#export_evaluation_report(evaluation_report)
#display(evaluation_report)

df_ratings_pd = load_ratings(surprise_format=False)
debug_users = df_ratings_pd[C.USER_ID_COL].drop_duplicates().sample(
    frac=1.0,
    random_state=1
)

df_ratings_pd = df_ratings_pd[
    df_ratings_pd[C.USER_ID_COL].isin(debug_users)
]

sp_ratings = Dataset.load_from_df(
    df_ratings_pd[[C.USER_ID_COL, C.ITEM_ID_COL, C.RATING_COL]],
    Reader(rating_scale=(0.5, 5))
)
precomputed_dict = precompute_information(df_ratings_pd)
evaluation_report = create_evaluation_report(EvalConfig(), sp_ratings, precomputed_dict, AVAILABLE_METRICS)
export_evaluation_report(evaluation_report)
display(evaluation_report)

Handling model ContentBased_ridge_cv
  Training split predictions
  - computing metric rmse
  - computing metric mae
Training loo predictions
  Training loo predictions (negative sampling, 100 candidates)
  Training full predictions
Handling model BPR
  Training split predictions


  0%|          | 0/100 [00:00<?, ?it/s]

  - computing metric rmse
  - computing metric mae
Training loo predictions


  0%|          | 0/100 [00:00<?, ?it/s]

  Training loo predictions (negative sampling, 100 candidates)


...

  0%|          | 0/20 [00:00<?, ?it/s]

  Training full predictions


  0%|          | 0/20 [00:00<?, ?it/s]

Handling model UserBased_Pearson_Natif
  Training split predictions
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
  - computing metric rmse
  - computing metric mae
Training loo predictions
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
  Training loo predictions (negative sampling, 100 candidates)
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
  Training full predictions
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Handling model UserBased_Tuned_Jaccard
  Training split predictions
  - computing metric rmse
  - computing metric mae
Training loo predictions
  Training loo predictions (negative sampling, 100 candidates)
  Training full predictions


,rmse,mae,hit_rate@5,hit_rate@10,hit_rate@20,ndcg@5,ndcg@10,ndcg@20,hit_rate@5[ns],hit_rate@10[ns],hit_rate@20[ns],ndcg@5[ns],ndcg@10[ns],ndcg@20[ns],coverage,miuf,ild
ContentBased_ridge_cv,0.744195,0.561289,0.015,0.024,0.041,0.009615,0.012420,0.016737,0.226,0.315,0.462,0.157875,0.186925,0.223890,0.359048,5.877882,0.642822
BPR,2.328037,2.056597,0.096,0.137,0.192,0.065064,0.078334,0.092044,0.661,0.782,0.866,0.495284,0.534774,0.556249,0.475449,3.274993,0.655373
BPR_Novelty,3.489391,3.334007,0.086,0.124,0.167,0.056892,0.069391,0.080050,0.625,0.751,0.817,0.469451,0.510775,0.527739,0.511732,3.627230,0.649654
iALS,2.768292,2.577183,0.016,0.028,0.051,0.009558,0.013314,0.018975,0.712,0.857,0.904,0.484896,0.532124,0.544223,0.421884,3.302279,0.690803
UserBased_Pearson_Natif,0.804929,0.611270,0.055,0.086,0.129,0.038147,0.047976,0.058781,0.115,0.222,0.430,0.062149,0.096564,0.148578,0.060662,1.251979,0.728214
UserBased_Tuned_Jaccard,0.835088,0.636121,0.050,0.075,0.113,0.032459,0.040484,0.050102,0.096,0.200,0.380,0.058116,0.091112,0.136178,0.051047,1.188285,0.727401


## Interpretation of Results

  ### Prerequisite Reading: Two Families of Models

  The models are not playing the same game, which means **not all metrics can be compared line-by-line**:

  * **Rating Prediction** (`ContentBased`, `UserBased`) → these produce a calibrated rating on a [0.5, 5] scale. RMSE/MAE is relevant for
  them.
  * **Implicit Ranking** (`BPR`, `BPR_Novelty`, `iALS`) → these produce an *uncalibrated* ranking score, not a rating. Their massive RMSE/MAE
  (2–3.5) is **not** a flaw: these models do not attempt to predict a rating, only to rank items properly. They must be evaluated on ranking,
  not RMSE.

  > **Note on `UserBased_Tuned_Jaccard`:** unlike `UserBased_Pearson_Natif` (Surprise's native `KNNBaseline`), this variant uses the custom
  `UserBased_tuned` class with a **hand-implemented Jaccard similarity** (`support / union`) that is **not available in Surprise**. It is 
  included to satisfy the requirement of using at least one non-Surprise similarity measure.
  
  ### 1. Rating Accuracy — RMSE / MAE (lower = better)

  * **ContentBased wins clearly**: RMSE **0.744**, MAE **0.561**. It is the best predictor of user taste.
  * UserBased follows (Pearson **0.805** / Tuned-Jaccard **0.835**). The native Pearson baseline edges out the custom Jaccard variant on 
  accuracy.
  * BPR / iALS / BPR_Novelty: very high values → **not interpretable** (see warning above).
  * *Conclusion: for a reliable "match %", content-based is the right choice.*

  ### 2. Ranking on Full Catalogue — hit_rate / NDCG (higher = better)

  Here, each positive item is ranked against **the entire catalogue** (a difficult task).
  
  * **BPR dominates**: hit_rate@10 = **0.137**, NDCG@10 = **0.078**.
  * BPR_Novelty is right behind (0.124 / 0.069): the novelty penalty costs a small amount of accuracy.
  * UserBased ≈ half as effective (hit_rate@10 ≈ **0.075–0.086**, Pearson > Tuned-Jaccard).
  * ContentBased and iALS are **weak** under this protocol (hit_rate@10 ≈ 0.02–0.03).

  ### 3. Ranking via Negative Sampling — `[ns]` metrics (1 positive vs 99 negatives)

  A much easier task → significantly higher scores. The ranking is **reversed** compared to section 2:

  * **iALS and BPR crush the competition**: iALS hit_rate@10 = **0.857** (best recall), BPR NDCG@10 = **0.535** (best ordering).
  * ContentBased (0.315) and UserBased (≈ **0.20–0.22**, Pearson > Tuned-Jaccard) lag far behind.
  * *Methodological Warning:* the dramatic gap between section 2 (full catalogue) and section 3 (`[ns]`) illustrates the well-known bias of
  sampled metrics (Krichene & Rendle, 2020) — a model can excel at distinguishing a positive from 99 random negatives while completely
  drowning when faced with the full catalogue. iALS is the textbook example of this.

  ### 4. Coverage & Novelty — coverage / MIUF (higher = broader catalogue / less popular items)

  * **Coverage**: BPR_Novelty (**0.512**) > BPR (0.475) > iALS (0.422) > Content (0.359) ≫ UserBased (**0.061 / 0.051**).
  * **MIUF (Novelty)**: ContentBased (**5.88**) recommends the most niche items; UserBased (**1.19–1.25**) recommends the most mainstream.
  * *Conclusion: UserBased recommends a tiny core of highly popular movies* — consistent with its **popularity re-ranking (α = 0.5)**. 
  Conversely, content-based and BPR_Novelty explore the long tail.

  ### 5. Intra-List Diversity — ILD (higher = more varied list)
  
  * **UserBased is the most diverse** within a single list (≈ **0.727–0.728**, Pearson and Tuned-Jaccard nearly tied), followed by iALS
  (0.691).
  * ContentBased is the least diverse (**0.643**): logical, since it groups movies with highly similar content features.
  
  ### Summary — Which Model for Which Role

  | Model | Main Strength | Coherent Role (Carousel) |
  | --- | --- | --- |
  | **ContentBased** | Best rating accuracy + highest novelty | "Recommended for You" + source of the match % |
  | **BPR** | Best ranking on full catalogue | "Discover Something New" |
  | **BPR_Novelty** | Max coverage/novelty (at the cost of slight accuracy) | Long-tail discovery |
  | **iALS** | Dominates `[ns]` ranking (positive/negatives separation) | "Top Picks For You" |
  | **UserBased (Pearson)** | Decent ratings, diverse lists, but narrow & highly popular catalogue | "Viewers Like You Also Watched" |
  | **UserBased (Tuned-Jaccard)** | Custom non-Surprise Jaccard similarity; close to Pearson but slightly weaker on every metric | Demonstrates a from-scratch similarity measure |
  

  **Key Takeaway:** No single model wins across all fronts — it is a trade-off between accuracy ↔ ranking ↔ novelty. This highlights the 
  benefit of combining multiple complementary carousels rather than relying on a single model. The two user-based variants also show that 
  Surprise's native Pearson baseline slightly outperforms a hand-built Jaccard similarity, while the latter satisfies the requirement of 
  implementing a similarity measure outside the library.